# Active particles cycling between two sites in 2D

**Setup.** A 2D Euclidean domain $\Omega \subset \mathbb{R}^2$ contains:

- a **source / charging station** at $\mathbf{x}_A = (0, 0)$,
- a **target / discharging station** at $\mathbf{x}_B \in \Omega$ (e.g. $\mathbf{x}_B = (L, 0)$).

A population of particles (think foragers, motor proteins, delivery drones) carries an internal *energy* or *cargo* variable. The goal is a model whose steady state shows a **persistent circulating current** $A \rightarrow B \rightarrow A$: uncharged particles flow toward $A$ to refuel, charged particles flow toward $B$ to deliver, and the cycle repeats.

Three structural features make this fundamentally a **non-equilibrium** problem, and rule out a single passive Fokker–Planck with one scalar potential:

1. The drift on a particle depends on its **internal state**, not just on $\mathbf{x}$.
2. There is a sustained **energy input at $A$** (chemical bath, light, food patch) which must be modelled explicitly or implicitly.
3. The steady state has nonzero probability **currents** — detailed balance is broken; no Gibbs measure $e^{-\beta U}$ exists.

Boundary conditions on $\Omega$ are an independent choice: reflecting walls (the natural analogue of the sloshing notebook), periodic box, or unbounded with strongly confining $U_A, U_B$. The options below are agnostic to that choice.


## Option 1 — Two-state Fokker–Planck (binary energy)

Closest analogue of the 1D run-and-tumble model. A particle is either **uncharged** ($\sigma = 0$) or **charged** ($\sigma = 1$); let

$$\rho_0(\mathbf{x}, t), \qquad \rho_1(\mathbf{x}, t), \qquad \rho := \rho_0 + \rho_1.$$

Each species feels its own attractive potential and shares the diffusion constant $D$:

$$\rho_0 \text{ feels } U_A(\mathbf{x}) \text{ (well at } A), \qquad \rho_1 \text{ feels } U_B(\mathbf{x}) \text{ (well at } B).$$

Charging and discharging are **localized reactions**,

$$k_{\uparrow}(\mathbf{x}) = \kappa_A\, \chi_A(\mathbf{x}), \qquad k_{\downarrow}(\mathbf{x}) = \kappa_B\, \chi_B(\mathbf{x}),$$

with $\chi_{A,B}$ smooth bumps (e.g. normalized Gaussians) concentrated near each site. The coupled system reads

$$
\boxed{\;
\begin{aligned}
\partial_t \rho_0 &= \nabla\!\cdot\!\bigl(\rho_0 \nabla U_A + D\,\nabla \rho_0\bigr) \;-\; k_{\uparrow}(\mathbf{x})\,\rho_0 \;+\; k_{\downarrow}(\mathbf{x})\,\rho_1, \\[2pt]
\partial_t \rho_1 &= \nabla\!\cdot\!\bigl(\rho_1 \nabla U_B + D\,\nabla \rho_1\bigr) \;+\; k_{\uparrow}(\mathbf{x})\,\rho_0 \;-\; k_{\downarrow}(\mathbf{x})\,\rho_1.
\end{aligned}\;}
$$

Key properties:

- **Mass conservation**: $\partial_t (\rho_0 + \rho_1) + \nabla\!\cdot(\mathbf{J}_0 + \mathbf{J}_1) = 0$. Reactions only shuffle mass between channels.
- The steady state carries **nonzero currents** $\mathbf{J}_\sigma = -\rho_\sigma \nabla U_\sigma - D \nabla \rho_\sigma$. In the cycle, $\mathbf{J}_1$ points $A \to B$ and $\mathbf{J}_0$ points $B \to A$ on average.
- The reaction terms in matrix form are
$$\partial_t\!\begin{pmatrix}\rho_0\\\rho_1\end{pmatrix} \supset \begin{pmatrix} -k_{\uparrow} & +k_{\downarrow} \\ +k_{\uparrow} & -k_{\downarrow}\end{pmatrix}\!\begin{pmatrix}\rho_0\\\rho_1\end{pmatrix}.$$

**Pros:** cheap (two scalar PDEs on the same 2D grid), reuses the upwind-FV / `solve_ivp` machinery from the 1D notebook with a per-channel drift.  
**Cons:** charging is binary and instantaneous — no notion of partial charge or charging time.


## Option 2 — Augmented Fokker–Planck on $(\mathbf{x}, e)$

Replace the binary $\sigma$ by a continuous **internal energy** $e \in [0, e_{\max}]$ and solve a single PDE for $p(\mathbf{x}, e, t)$ on the 3D state space:

$$
\boxed{\;
\partial_t p \;=\; -\nabla_{\mathbf{x}}\!\cdot\!\bigl[\mathbf{v}(\mathbf{x}, e)\,p\bigr] \;+\; D\,\nabla_{\mathbf{x}}^2 p \;-\; \partial_e\!\bigl[g(\mathbf{x}, e)\,p\bigr] \;+\; \tilde D\, \partial_e^2 p .
\;}
$$

The energy-dependent **drift** interpolates smoothly between the two attractors,

$$\mathbf{v}(\mathbf{x}, e) \;=\; -\bigl(1 - h(e)\bigr)\,\nabla U_A(\mathbf{x}) \;-\; h(e)\,\nabla U_B(\mathbf{x}),$$

with switching function

$$h(e) = \tfrac{1}{2}\Bigl[1 + \tanh\!\bigl(\tfrac{e - e^\star}{\Delta e}\bigr)\Bigr] \;\in [0,1].$$

So an uncharged particle ($e \approx 0$) is pulled toward $A$, a charged one ($e \approx e_{\max}$) toward $B$, with a smooth handover at $e \approx e^\star$.

The **energy dynamics** is local in $\mathbf{x}$:

$$g(\mathbf{x}, e) \;=\; \underbrace{\gamma_A\, \chi_A(\mathbf{x})\,(e_{\max} - e)}_{\text{charging near }A} \;-\; \underbrace{\gamma_B\, \chi_B(\mathbf{x})\, e}_{\text{discharging near }B} \;-\; \underbrace{\lambda\, e}_{\text{leakage everywhere}}.$$

Boundary conditions in energy: **reflective** at $e = 0$ and $e = e_{\max}$ if particles are never destroyed; **absorbing** at $e = 0$ if a fully-drained particle is removed from the bath.

**Pros:** smooth, physically interpretable. Lets you ask quantitative questions like "what fraction of the input energy is dissipated in transit?".  
**Cons:** genuine 3D PDE — discretisation needs $N_x \times N_y \times N_e$ cells, plus an additional CFL constraint along $e$ if $g$ is large.


## Option 3 — Agent-based (Langevin) baseline

Simulate $N$ particles individually. Each $i = 1, \ldots, N$ carries $(\mathbf{x}_i, e_i)$ and follows

$$
\mathrm{d}\mathbf{x}_i \;=\; \mathbf{v}\bigl(\mathbf{x}_i,\, e_i\bigr)\,\mathrm{d}t \;+\; \sqrt{2D}\,\mathrm{d}\mathbf{W}_i,
$$

$$
\mathrm{d}e_i \;=\; g\bigl(\mathbf{x}_i,\, e_i\bigr)\,\mathrm{d}t \;+\; \sqrt{2\tilde D}\,\mathrm{d}\widetilde W_i ,
$$

with the same $\mathbf{v}$ and $g$ as Option 2, **or** with Poisson jumps in $e$ for the binary picture of Option 1:

$$\mathbb{P}\bigl[\sigma_i:\,0 \to 1 \text{ in } [t, t+\mathrm{d}t]\bigr] = k_{\uparrow}(\mathbf{x}_i)\,\mathrm{d}t, \qquad \mathbb{P}\bigl[\sigma_i:\,1 \to 0\bigr] = k_{\downarrow}(\mathbf{x}_i)\,\mathrm{d}t .$$

The empirical density

$$\hat p_N(\mathbf{x}, e, t) \;=\; \tfrac{1}{N}\sum_{i=1}^N \delta\bigl(\mathbf{x} - \mathbf{x}_i(t)\bigr)\,\delta\bigl(e - e_i(t)\bigr)$$

converges as $N \to \infty$ (and at fixed $t$) to the Fokker–Planck solution of Option 2 — so the agent model is a stochastic realization of the same physics.

**Pros:** trivial to simulate (vectorised Euler–Maruyama in NumPy), gives intuition fast, and accommodates **interactions** (excluded volume, density-dependent speed, alignment) without rewriting the PDE.  
**Cons:** stochastic; statistics need many particles or many runs.


## Option 4 — Active matter: state-dependent self-propulsion

Borrow the active-matter vocabulary directly. Each particle has position $\mathbf{x}$, orientation $\hat{\mathbf{n}}(\theta) = (\cos\theta, \sin\theta)$, and an internal energy $e$. The self-propulsion **speed** and the **steering torque** both depend on the state:

$$
\mathrm{d}\mathbf{x} \;=\; v_0(e)\,\hat{\mathbf{n}}(\theta)\,\mathrm{d}t \;+\; \sqrt{2 D_t}\,\mathrm{d}\mathbf{W},
$$

$$
\mathrm{d}\theta \;=\; \mu(\mathbf{x}, e)\,\mathrm{d}t \;+\; \sqrt{2 D_r}\,\mathrm{d}W_\theta ,
$$

with a biasing torque that aligns the particle with the bearing toward $A$ when uncharged and toward $B$ when charged:

$$\mu(\mathbf{x}, e) \;=\; -\bigl(1 - h(e)\bigr)\,\sin\!\bigl(\theta - \theta_A(\mathbf{x})\bigr) \;-\; h(e)\,\sin\!\bigl(\theta - \theta_B(\mathbf{x})\bigr),$$

where $\theta_{A,B}(\mathbf{x}) = \arg(\mathbf{x}_{A,B} - \mathbf{x})$ is the bearing toward each station.

The corresponding Fokker–Planck lives on the **4D state space** $(\mathbf{x}, \theta, e)$. This is the close relative of **Active Brownian Particles** and **chemotactic bacteria** models, and is the right level of description if you care about:

- persistent runs and finite turning rates,
- **wall accumulation** (active particles pin against confining boundaries),
- **motility-induced phase separation** when $v_0$ also depends on local density.

**Pros:** physically motivated for swimmers, motor proteins, robotic agents.  
**Cons:** extra orientation variable bumps state-space dimension to 4; needs care with periodicity in $\theta$.


## Option 5 — Coarse-grained: nonequilibrium steady-state currents

Forget the microscopic equations and ask only about the **steady-state cycle**. Treat the system as a two-node network $\{A, B\}$ with two transport channels (charged outbound, uncharged return). The steady-state mass current along the cycle is a single scalar $J$.

In the linear-response regime,

$$J \;=\; \mathcal{L}\,\bigl(\mu_A - \mu_B\bigr),$$

where $\mu_A - \mu_B$ plays the role of a **chemical-potential drop** sustained by the energy source at $A$, and $\mathcal{L}$ is an Onsager conductance set by particle mobility, noise level, and the geometry of the wells. The energy input rate is

$$\dot Q \;=\; J\,(\mu_A - \mu_B),$$

formally identical to the electrical power dissipated by a current through a battery.

**Use case:** a back-of-envelope sanity check for any of the microscopic models above. Pick parameters, measure $J$ from the simulation, and see whether the scaling with $D$, $\kappa_A$, $\kappa_B$, and the geometric distance $|\mathbf{x}_A - \mathbf{x}_B|$ matches what this picture predicts.


## Comparison and a suggested progression

| Model | Per-particle state | PDE dimension | Partial charge | Hardest new term |
|---|---|---|---|---|
| **1.** Two-state FP | $(\mathbf{x},\,\sigma\!\in\!\{0,1\})$ | 2D $\times$ 2 channels | no | localized reaction rates |
| **2.** Continuous-$e$ FP | $(\mathbf{x},\, e)$ | 3D | yes | energy flux $\partial_e[g\,p]$ |
| **3.** Langevin SDE | $(\mathbf{x}_i,\, e_i)$, $i=1..N$ | none (SDE) | yes | many-particle bookkeeping |
| **4.** State-dependent ABP | $(\mathbf{x},\, \theta,\, e)$ | 4D | yes | orientation diffusion |
| **5.** Coarse-grained network | nodes $A,\, B$ | network | aggregate only | parameter identification |

**Suggested progression:**

1. Start with **Option 3** (Langevin with binary energy): minimal code, the cycle becomes visible after a few thousand steps with $N \sim 10^3$ particles.
2. Coarse-grain to **Option 1** (two-state FP) to verify that the same circulating current emerges in the continuum limit. Reuses the upwind-FV / `solve_ivp` machinery from the 1D notebook, with the spatial operator extended to 2D.
3. If the question is *quantitative* energetics (transit losses, charging time distributions), upgrade to **Option 2**.
4. Move to **Option 4** only if persistent orientation matters physically — i.e. for real swimmers, run-and-tumble bacteria, or motor proteins on cytoskeletal tracks.
